# 01 — البيئة وقاعدة البيانات (Environment & DB sanity)

**الهدف:** التأكد إن الأساس سليم قبل أي اختبار تاني: الـ Python والمكتبات، الـ Settings، PostgreSQL + pgvector، شكل الجداول، الـ Foreign Keys، الـ Routes، وموديل الـ Embedding.

> لو الـ notebook ده فيه FAIL، الـ notebooks التانية غالباً هتفشل بسببه — صلّحه الأول.

**طريقة التشغيل:** `Kernel → Restart & Run All`. الـ notebooks دي بتشتغل على نفس الـ `app` مباشرة (FastAPI TestClient) ومع Postgres حقيقي.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
sys.path.insert(0, str(pathlib.Path.cwd() / "notebooks"))
from nb_utils import *
env = setup()
print(env)
rec = Recorder("01_environment_and_db")

## 1) البيئة والمكتبات

In [ ]:
import importlib.metadata as md
pkgs = ["fastapi", "starlette", "uvicorn", "sqlalchemy", "pydantic", "pydantic-settings", "pgvector",
        "psycopg2-binary", "psycopg", "alembic", "prometheus-client", "sentence-transformers", "torch",
        "httpx", "pandas", "matplotlib", "pyjwt", "python-jose", "psutil"]
rows = []
for p in pkgs:
    try:
        rows.append((p, md.version(p)))
    except md.PackageNotFoundError:
        rows.append((p, "— not installed"))
display(pd.DataFrame(rows, columns=["package", "version"]))

with rec.case("Python >= 3.10", "env") as c:
    assert sys.version_info >= (3, 10), sys.version
    c.note(platform.python_version(), platform.system())

for must in ("fastapi", "sqlalchemy", "pgvector", "prometheus-client", "sentence-transformers", "httpx"):
    with rec.case(f"package installed: {must}", "env"):
        md.version(must)

## 2) الإعدادات (Settings)
بنطبع الـ settings مع إخفاء أي secret.

In [ ]:
S, data = None, {}
with rec.case("import settings", "config") as c:
    try:
        from helpers import settings as S
    except ImportError:
        from helpers.config import settings as S
    if callable(S) and not hasattr(S, "model_dump"):
        S = S()
    data = S.model_dump() if hasattr(S, "model_dump") else {k: getattr(S, k) for k in dir(S) if k.isupper()}
    c.note(f"{len(data)} settings loaded")
if data:
    display(pd.DataFrame([(k, mask(k, v)) for k, v in sorted(data.items())], columns=["setting", "value"]))

with rec.case("LLM provider is 'fake' during tests (no real API calls / cost)", "config") as c:
    prov = next((str(v) for k, v in data.items() if "provider" in k.lower()), None)
    if prov is None:
        raise Skip("no *provider* setting found")
    if "fake" not in prov.lower():
        raise Warn(f"provider={prov}. Set CONFIG['ENV_OVERRIDES'] in nb_utils.py to the right env var name")
    c.note(prov)

with rec.case("JWT secret is not a placeholder", "config") as c:
    sec = next((str(v) for k, v in data.items() if re.search("secret", k, re.I) and v), None)
    if sec is None:
        raise Skip("no secret setting found")
    if len(sec) < 32 or sec.lower() in {"secret", "changeme", "change-me", "your-secret-key"}:
        raise Warn("secret looks weak/short - fine for dev, never for production")

## 3) PostgreSQL + pgvector

In [ ]:
from sqlalchemy import text, inspect as sa_inspect
DBNAME = None
with rec.case("connect to PostgreSQL", "database") as c:
    with get_session() as s:
        version = s.execute(text("select version()")).scalar()
        DBNAME = s.execute(text("select current_database()")).scalar()
    c.note(f"db={DBNAME}", version.split(",")[0])

with rec.case("running against a dedicated TEST database", "database") as c:
    need(DBNAME)
    if "test" not in DBNAME.lower():
        raise Warn(f"current DB is '{DBNAME}'. The notebooks create users/workspaces/files: "
                   "better point DATABASE_URL to a separate `rag_test` database")

with rec.case("pgvector extension installed", "database") as c:
    with get_session() as s:
        row = s.execute(text("select extversion from pg_extension where extname='vector'")).first()
    assert row, "extension 'vector' missing (CREATE EXTENSION vector;)"
    c.note("version", row[0])

In [ ]:
tbl_rows = []
with rec.case("tables exist", "database") as c:
    with get_session() as s:
        insp = sa_inspect(s.get_bind())
        tables = sorted(insp.get_table_names())
        assert tables, "no tables found - run migrations / create_all first"
        for t in tables:
            n = s.execute(text(f'select count(*) from "{t}"')).scalar()
            tbl_rows.append(dict(table=t, columns=len(insp.get_columns(t)), rows=n,
                                 indexes=len(insp.get_indexes(t)), foreign_keys=len(insp.get_foreign_keys(t))))
    c.note(f"{len(tables)} tables")
display(pd.DataFrame(tbl_rows))

VEC = []
with rec.case("vector column exists with the right dimension", "database") as c:
    with get_session() as s:
        VEC = find_vector_columns(s)
    assert VEC, "no pgvector column found"
    dims = {t: int(re.search(r"\((\d+)\)", ty).group(1)) for t, col, ty in VEC if re.search(r"\((\d+)\)", ty)}
    assert all(d == CONFIG["EMBED_DIM"] for d in dims.values()), f"dims={dims}, expected {CONFIG['EMBED_DIM']}"
    c.note(VEC)

with rec.case("vector (HNSW/IVFFlat) index exists", "database") as c:
    with get_session() as s:
        idx = s.execute(text("select tablename, indexname from pg_indexes "
                             "where indexdef ilike '%hnsw%' or indexdef ilike '%ivfflat%'")).all()
    if not idx:
        raise Warn("no ANN index: every search is a sequential scan (OK for dev, slow at scale)")
    c.note([i[1] for i in idx])

with rec.case("alembic migrations table present", "database") as c:
    with get_session() as s:
        ok = "alembic_version" in sa_inspect(s.get_bind()).get_table_names()
    if not ok:
        raise Warn("no alembic_version table - schema was probably created with create_all")

In [ ]:
orph = []
with rec.case("no orphan rows (foreign-key integrity)", "database") as c:
    with get_session() as s:
        insp = sa_inspect(s.get_bind())
        for t in insp.get_table_names():
            for fk in insp.get_foreign_keys(t):
                if len(fk["constrained_columns"]) != 1:
                    continue
                col, rt, rc = fk["constrained_columns"][0], fk["referred_table"], fk["referred_columns"][0]
                n = s.execute(text(f'select count(*) from "{t}" a left join "{rt}" b on a."{col}"=b."{rc}" '
                                   f'where a."{col}" is not null and b."{rc}" is null')).scalar()
                orph.append(dict(table=t, fk_column=col, parent=rt, orphans=n,
                                 on_delete=(fk.get("options") or {}).get("ondelete")))
    bad = [o for o in orph if o["orphans"]]
    assert not bad, f"orphans found: {bad}"
    c.note(f"{len(orph)} foreign keys checked")
display(pd.DataFrame(orph))

with rec.case("chunks are deleted automatically with their file (ON DELETE CASCADE)", "database") as c:
    with get_session() as s:
        ctable, _ = chunk_table(s)
    fks = [o for o in orph if o["table"] == ctable]
    assert fks, f"no FK found on '{ctable}'"
    if not any((o["on_delete"] or "").upper() == "CASCADE" for o in fks):
        raise Warn(f"FKs on {ctable} have no ON DELETE CASCADE -> orphan chunks / delete must be done in code")

## 4) التطبيق والـ Routes
هنا بنعرف الـ Routes اللي الـ notebooks هتستخدمها. أي route بيطلع **NOT FOUND** عدّل الـ regex بتاعه في `CONFIG['ROUTES']` جوه `nb_utils.py`.

In [ ]:
app, routes = None, {}
with rec.case("import FastAPI app", "app") as c:
    app = get_app()
    c.note(f"{len(app.routes)} routes registered")
need(app)
routes = resolve_routes(app)
display(pd.DataFrame([(k, m, p or "— NOT FOUND —") for k, (m, p) in routes.items()],
                     columns=["key", "method", "resolved OpenAPI path"]))
for k, (m, p) in routes.items():
    rec.check(f"route resolved: {k}", p is not None, f"{m} {p}", "app", level="warn")

spec = app.openapi()
ops = []
for p, methods in spec["paths"].items():
    for m, op in methods.items():
        if m.lower() in ("get", "post", "put", "patch", "delete"):
            ops.append(dict(method=m.upper(), path=p, tags=",".join(op.get("tags", [])), summary=op.get("summary", "")))
rec.check("OpenAPI schema generated", len(ops) > 0, f"{len(ops)} operations", "app")
display(pd.DataFrame(ops))

c:\Users\Options Store\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:223: FutureWarning: You are using a non-supported Python version (3.10.11). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.api_core.
  warnings.warn(message, FutureWarning)
C:\Users\Options Store\AppData\Local\Temp\ipykernel_10932\133491375.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/l

## 5) موديل الـ Embedding
نتأكد إن الموديل بيتحمّل وإن الـ dimension بتاعه = الـ dimension بتاع عمود الـ vector في الداتابيز.

In [ ]:
model = None
with rec.case("load embedding model", "embedding") as c:
    from sentence_transformers import SentenceTransformer
    name = getattr(S, "EMBEDDING_MODEL_NAME", None) or getattr(S, "EMBEDDING_MODEL", None) or CONFIG["EMBEDDING_MODEL"]
    t0 = time.perf_counter()
    model = SentenceTransformer(name)
    c.note(name, f"loaded in {time.perf_counter() - t0:.1f}s")

with rec.case(f"model output dimension == {CONFIG['EMBED_DIM']} == DB column", "embedding") as c:
    need(model)
    v = model.encode(["passage: hello world"])
    assert v.shape == (1, CONFIG["EMBED_DIM"]), f"model dim={v.shape[1]}, config/DB dim={CONFIG['EMBED_DIM']}"
    c.note(f"max_seq_length={model.max_seq_length} tokens")

## 6) الملخص

In [ ]:
rec.summary()
rec.table()
rec.save()